In [145]:
import tensorflow as tf 
from tensorflow.keras.models import load_model
import pickle 
import pandas as pd
import numpy as np


In [146]:
### load all the train models like ann, scalar and pickle file for the columns
model = load_model('model.h5')
with open('geo.pkl', 'rb') as f:
    label_encoder_geo = pickle.load(f)
with open('label_encoder_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)


/Users/ritvik_singh/Study/LLM_COURSE/LLM_Course/.venv/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [147]:
#example input data
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [148]:
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df.head()


/Users/ritvik_singh/Study/LLM_COURSE/LLM_Course/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [149]:
input_df=pd.DataFrame(input_data, index=[0])
input_df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [150]:
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df.head() 

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [151]:
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]])

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=label_encoder_geo.get_feature_names_out(['Geography'])
)

/Users/ritvik_singh/Study/LLM_COURSE/LLM_Course/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [152]:
input_df=pd.concat([input_df, geo_encoded_df], axis=1)
input_df.head() 


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [153]:
input_df = input_df.drop(['Geography'], axis=1)

In [154]:
#scaling the input data
input_scaled= scaler.transform(input_df)
input_scaled


array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.41421356,
         0.        , -1.41421356]])

In [155]:
#predict churn 
prediction = model.predict(input_scaled)
print(prediction)


1/1 [==============================] - 0s 44ms/step
[[5.7977495e-10]]


In [156]:
print(input_df)
print(input_df.dtypes)
print(input_df.isnull().sum())

   CreditScore  Gender  Age  Tenure  Balance  NumOfProducts  HasCrCard  \
0          600       1   40       3    60000              2          1   

   IsActiveMember  EstimatedSalary  Geography_France  Geography_Germany  \
0               1            50000               1.0                0.0   

   Geography_Spain  
0              0.0  
CreditScore            int64
Gender                 int64
Age                    int64
Tenure                 int64
Balance                int64
NumOfProducts          int64
HasCrCard              int64
IsActiveMember         int64
EstimatedSalary        int64
Geography_France     float64
Geography_Germany    float64
Geography_Spain      float64
dtype: object
CreditScore          0
Gender               0
Age                  0
Tenure               0
Balance              0
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
Geography_France     0
Geography_Germany    0
Geography_Spain      0
dtype: int64


In [157]:
print(input_scaled)

[[-0.53598516  0.91324755  0.10479359 -0.69539349 -0.25781119  0.80843615
   0.64920267  0.97481699 -0.87683221  1.41421356  0.         -1.41421356]]


In [158]:
print("Training columns:")
print(scaler.feature_names_in_)

print("\nInput columns:")
print(input_df.columns)

Training columns:
['CreditScore' 'Gender' 'Age' 'Tenure' 'Balance' 'NumOfProducts'
 'HasCrCard' 'IsActiveMember' 'EstimatedSalary' 'Geography_France'
 'Geography_Germany' 'Geography_Spain']

Input columns:
Index(['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
       'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France',
       'Geography_Germany', 'Geography_Spain'],
      dtype='object')


In [159]:
import numpy as np
print(np.isfinite(input_scaled))

[[ True  True  True  True  True  True  True  True  True  True  True  True]]


In [160]:
input_df = pd.DataFrame(input_data, index=[0])

# Encode gender
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

# Encode geography
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]])
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=label_encoder_geo.get_feature_names_out(['Geography'])
)

input_df = pd.concat([input_df, geo_encoded_df], axis=1)

# Drop original column
input_df = input_df.drop(['Geography'], axis=1)

# Add missing columns
for col in scaler.feature_names_in_:
    if col not in input_df.columns:
        input_df[col] = 0

# Correct order
input_df = input_df[scaler.feature_names_in_]

# Scale
input_scaled = scaler.transform(input_df)

# Predict
prediction = model.predict(input_scaled)

print(prediction)

/Users/ritvik_singh/Study/LLM_COURSE/LLM_Course/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


1/1 [==============================] - 0s 16ms/step
[[5.7977495e-10]]


In [161]:
print(input_df)
print(input_df.dtypes)
print(input_scaled)

   CreditScore  Gender  Age  Tenure  Balance  NumOfProducts  HasCrCard  \
0          600       1   40       3    60000              2          1   

   IsActiveMember  EstimatedSalary  Geography_France  Geography_Germany  \
0               1            50000               1.0                0.0   

   Geography_Spain  
0              0.0  
CreditScore            int64
Gender                 int64
Age                    int64
Tenure                 int64
Balance                int64
NumOfProducts          int64
HasCrCard              int64
IsActiveMember         int64
EstimatedSalary        int64
Geography_France     float64
Geography_Germany    float64
Geography_Spain      float64
dtype: object
[[-0.53598516  0.91324755  0.10479359 -0.69539349 -0.25781119  0.80843615
   0.64920267  0.97481699 -0.87683221  1.41421356  0.         -1.41421356]]


In [162]:
import numpy as np

for layer in model.layers:
    weights = layer.get_weights()
    for w in weights:
        print(np.isnan(w).any())

False
False
False
False
False
False


In [163]:
prediction_proba = prediction[0][0]

In [164]:
prediction_proba

5.7977495e-10

In [165]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")       

The customer is not likely to churn.
